In [26]:
import pandas as pd
import sys
sys.executable
from urllib.parse import urlencode
import requests
from pprint import pprint
from math import sin, cos, sqrt, atan2, radians
from global_land_mask import globe
import math
import datetime
import xlsxwriter
import datetime
from helper import *
today = datetime.date.today()

In [27]:

from utils import check_routes, get_waypoint0, get_waypoint1
from decouple import config

from tqdm import tqdm


# read here_key.txt
with open('here_key.txt', 'r') as f:
    here_key = f.read()

In [28]:
# Pull King Elvis files from Sharepoint to here, need to have been reviewed by secondary reviewers

gcrta_ke = pd.read_excel('ke_files/2023_GCRTA_KINGElvis.xlsx', sheet_name = 'Elvis_Review')
hrtva_ke  = pd.read_excel('ke_files/HRT_KINGElvis.xlsx', sheet_name = 'Elvis_Review')
lacmta_ke = pd.read_excel('ke_files/LACMTA_KINGElvis.xlsx', sheet_name = 'Elvis_Review')
bct_ke = pd.read_excel('ke_files/BCT_KINGElvis.xlsx', sheet_name = 'Elvis_Review')
wake_ke = pd.read_excel('ke_files/TRIANGLE_WAKE_KINGElvis.xlsx', sheet_name = 'Elvis_Review')
victor_ke = pd.read_excel('ke_files/VICTOR_KINGElvis.xlsx', sheet_name = 'Elvis_Review')
cota_ke = pd.read_excel('ke_files/COTA_KINGElvis.xlsx', sheet_name = 'Elvis_Review')


/home/austin/Documents/Repositories/etc_transit/gtfs/route_info_materials/BROWARD_FL/KingElvisAAReview/etc3/lib/python3.9/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
/home/austin/Documents/Repositories/etc_transit/gtfs/route_info_materials/BROWARD_FL/KingElvisAAReview/etc3/lib/python3.9/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
/home/austin/Documents/Repositories/etc_transit/gtfs/route_info_materials/BROWARD_FL/KingElvisAAReview/etc3/lib/python3.9/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
/home/austin/Documents/Repositories/etc_transit/gtfs/route_info_materials/BROWARD_FL/KingElvisAAReview/etc3/lib/python3.9/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Conditional Formatting extension is not

In [29]:
source_dir = '/home/austin/Documents/Repositories/etc_transit/gtfs/route_info_materials/Hampton Roads - HRT/HereAPIPOC/HRTVA_AUTOMATION/tables'
dest_dir = '/home/austin/Documents/Repositories/etc_transit/gtfs/route_info_materials/BROWARD_FL/KingElvisAAReview/tables'

# Call the sync_folders function at the beginning of your secondary script
sync_folders(source_dir, dest_dir)

In [30]:
gcrta_df = pd.read_csv('tables/elvisawsclevelandrta2023obweekday_export_odbc_raw df check.csv')
hrtva_df = pd.read_csv('tables/elvishrtva2023obweekday_export_odbc_raw df check.csv')
lacmta_df = pd.read_csv('tables/elvislacmta2023obweekday_export_odbc_raw df check.csv')
bct_df = pd.read_csv('tables/elvisbroward2023obweekday_export_odbc_raw df check.csv')
wake_df = pd.read_csv('tables/elviswakenc2023obweekday_export_odbc_raw df check.csv')
victor_df = pd.read_csv('tables/elvisvictorvalley2023obweekday_export_odbc_raw df check.csv')
cota_df = pd.read_csv('tables/elviscota2023obweekday_export_odbc_raw df check.csv')

/tmp/ipykernel_263560/3523146005.py:1: DtypeWarning: Columns (17,28,29,30,31,32,45,46,47,48,49,50,53,54,61,62,63,64,66,67,68,107,117,118,119,120,141,144,145,146,151,153,161,171,174,177,192,193,194,196,197,202,204,211,218,219,223,237,238,242,255,257,259,268,274,277,284,285,289,291,293,306,308,310,320,321,322,323,324,325,326,327,328,329,330,331,333,334,337,338,339,340,341,342,343,344,345,346,347,348,350,351,355,356,358,360,362,364,365,367,368,372,373,375,377,379,381,382,384,385,393,395,404,410,412,421,422,423,424,425,426,427,428,429,430,431,432,433,435,436,438,439,440,441,442,443,444,445,446,447,448,449,450,452,453,455,456,457,458,459,460,462,463,464,465,466,467,469,470,473,474,475,477,479,481,482,483,484,486,487,550,551,552,554,555,563,564,569,589,591,620,621,622,623,624,625,626,627,629,639,661,663,673,687,689,691,693,695,697,699,701,703,705,707,709,711,713,715,717,719,721,723,725,727,729,731,733,735,737,739,741,743,745,747,749,751,753,755,757,759,761,763,771,773,790,793,853,854,855,856

In [31]:
# A function to remove rows where INTERV_INIT==999
def remove_test_records(df):
    df = df[df['INTERV_INIT']!=999]
    return df

# Apply remote_test_records function to all dataframes and ke's 
gcrta_df = remove_test_records(gcrta_df)
hrtva_df = remove_test_records(hrtva_df)
lacmta_df = remove_test_records(lacmta_df)
bct_df = remove_test_records(bct_df)
wake_df = remove_test_records(wake_df)
victor_df = remove_test_records(victor_df)

grcta_ke = remove_test_records(gcrta_ke)
hrtva_ke = remove_test_records(hrtva_ke)
lacmta_ke = remove_test_records(lacmta_ke)
bct_ke = remove_test_records(bct_ke)
wake_ke = remove_test_records(wake_ke)
victor_ke = remove_test_records(victor_ke)


In [32]:
def haversine_distance(coord1, coord2):
    R = 3958.8  # Radius of the Earth in miles
    try:
        lat1, lon1 = radians(coord1[0]), radians(coord1[1])
    except:
        return 10000000
    try:
        lat2, lon2 = radians(coord2[0]), radians(coord2[1])
    except:
        lat2, lon2 = coord2.split(',')
        lat2, lon2 = radians(float(lat2)), radians(float(lon2))
    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    distance = R * c
    print(distance)
    return distance

# Function to removal marked records


# Get records where Final_Usage equals 'Remove' 
def get_removal_marked_records(df: pd.DataFrame) -> pd.DataFrame:
    """
    Get rows from a DataFrame where 'Final_Usage' or 'FINAL_USAGE' is marked as 'Remove' or 'remove'.
    
    Args:
        df (pd.DataFrame): Input DataFrame
    
    Returns:
        pd.DataFrame: DataFrame containing the rows that should be removed
    """
    # Using the 'isin' function to check multiple values at once and casefold for case-insensitive matching
    columns_to_check = ['Final_Usage', 'FINAL_USAGE']
    
    for column in columns_to_check:
        if column in df.columns:
            return df[df[column].apply(lambda x: str(x).casefold()) == 'remove']
    
    return pd.DataFrame()

In [33]:
gcrta_removals = get_removal_marked_records(gcrta_ke)
hrtva_removals = get_removal_marked_records(hrtva_ke)
lacmta_removals = get_removal_marked_records(lacmta_ke)
bct_removals = get_removal_marked_records(bct_ke)
wake_removals = get_removal_marked_records(wake_ke)
victor_removals = get_removal_marked_records(victor_ke)
cota_removals = get_removal_marked_records(cota_ke)

In [34]:
def filter_df_by_ids(df, removals_df, column_name='id'):
    """
    Filters a DataFrame based on a list of IDs from another DataFrame.

    Parameters:
        df (DataFrame): The DataFrame to be filtered.
        removals_df (DataFrame): The DataFrame containing IDs to filter out.
        column_name (str): The name of the column containing the IDs.

    Returns:
        DataFrame: A filtered DataFrame containing only the rows with IDs in removals_df.
    """
    removal_ids = removals_df[column_name]
    filtered_df = df[df[column_name].isin(removal_ids)]
    return filtered_df

In [35]:
# Apply filter_df_by_ids to each DataFrame
gcrta_removals = filter_df_by_ids(gcrta_df, gcrta_removals)
hrtva_removals = filter_df_by_ids(hrtva_df, hrtva_removals)
lacmta_removals = filter_df_by_ids(lacmta_df, lacmta_removals)
bct_removals = filter_df_by_ids(bct_df, bct_removals)
wake_removals = filter_df_by_ids(wake_df, wake_removals)
victor_removals = filter_df_by_ids(victor_df, victor_removals)
cota_removals = filter_df_by_ids(cota_df, cota_removals)

In [36]:
# ALL CHECKS TO RUN
# Look for values with outside agency transfers - Maybe use interline?
#   - Can I get route names with HereAPI? Interline may be best here
# Check Place_name (ORIGIN_NAME or DESTIN_NAME) against reverse geocode description?
#

# FINAL REVIEWER BECOMES AUSTIN F

In [37]:
# def get_boarding_points(cord_df, here_key):
#     status_list = []
#     here_boarding_points = []  # List to store boarding points
#     print("Checking Routes...")
    
#     for index, cord in tqdm(cord_df.iterrows(), total=cord_df.shape[0]):
#         base_url = "https://router.hereapi.com/v8/routes?"
#         waypoint0 = get_waypoint0(cord)
#         waypoint1 = get_waypoint1(cord)
        
#         params = {
#             'apiKey': here_key,
#             'origin': waypoint0,
#             'destination': waypoint1,
#             'transportMode': 'bus'
#         }

#         url_with_params = base_url + urlencode(params)
        
#         # Add via waypoints if available
#         for col in cord_df.columns:
#             if '_LAT' in col and col not in ['ORIGIN_ADDRESS_LAT', 'DESTIN_ADDRESS_LAT']:
#                 long_col = col.replace('_LAT', '_LONG')  # Find the corresponding longitude column
#                 if not pd.isna(cord.get(col)) and not pd.isna(cord.get(long_col)):
#                     if cord[col] not in [',', ''] and cord[long_col] not in [',', '']:
#                         via_waypoint = f"{cord[col]},{cord[long_col]}"
#                         url_with_params += f"&via={via_waypoint}"

#         response = requests.get(url_with_params)
#         response_json = response.json()
#         #pprint(response_json)

#         # Extract the second pair of coordinates if it exists
#         try:
#             second_pair = response_json['routes'][0]['sections'][1]['arrival']['place']['location']['lat'], response_json['routes'][0]['sections'][1]['arrival']['place']['location']['lng']
#             here_boarding_points.append(second_pair)
#         except (KeyError, IndexError):
#             here_boarding_points.append(None)  # Append None if the second pair is not available
    
#     # Add the boarding points to the DataFrame
#     cord_df['here_boarding_point'] = here_boarding_points
    
#     return cord_df
 

def get_boarding_points(cord_df):
    here_boarding_points = []  # List to store boarding points
    print("Processing Routes...")
    
    for index, cord in cord_df.iterrows():
        
        # Initialize to None, in case the second pair doesn't exist
        second_pair = None

        # Search for second pair of coordinates if available
        lat_cols = [col for col in cord_df.columns if '_LAT' in col and col not in ['ORIGIN_ADDRESS_LAT', 'DESTIN_ADDRESS_LAT']]
        long_cols = [col.replace('_LAT', '_LONG') for col in lat_cols]
        
        if len(lat_cols) > 1:
            lat = cord.get(lat_cols[1])
            long = cord.get(long_cols[1])
            
            if not pd.isna(lat) and not pd.isna(long):
                if lat not in [',', ''] and long not in [',', '']:
                    second_pair = (lat, long)

        here_boarding_points.append(second_pair)
    
    # Add the boarding points to the DataFrame
    cord_df['here_boarding_point'] = here_boarding_points
    
    return cord_df


   

In [38]:
#bp_df = get_boarding_points(hrtva_removals, here_key)

In [39]:
def evaluate_removal_conditions(df):
    # Initialize empty lists to store values for each new column
    REVIEW_REVIEWER = []
    REVIEW_USAGE = []
    FLAG_ALL_EQUAL = []
    FLAG_POSS_HD = []
    FLAG_POSS_TRAN = []
    FLAG_WATER_ORIGIN = []  # Flag for origin in water
    FLAG_WATER_DESTIN = []  # Flag for destination in water
    def sanitize_coordinates(lat, lon):
        # Ensure latitude is within [-90, 90]
        lat = max(min(lat, 90), -90)
        # Ensure longitude is within [-180, 180]
        lon = (lon + 180) % 360 - 180
        return lat, lon

    for index, row in df.iterrows():
        HOME = row['HOME_ADDRESS_LAT'], row['HOME_ADDRESS_LONG']
        # Check for NaN values in ORIGIN and DESTIN
        ORIGIN = None if math.isnan(row['ORIGIN_ADDRESS_LAT']) or math.isnan(row['ORIGIN_ADDRESS_LONG']) else (row['ORIGIN_ADDRESS_LAT'], row['ORIGIN_ADDRESS_LONG'])
        DESTIN = None if math.isnan(row['DESTIN_ADDRESS_LAT']) or math.isnan(row['DESTIN_ADDRESS_LONG']) else (row['DESTIN_ADDRESS_LAT'], row['DESTIN_ADDRESS_LONG'])
        
        BOARDING = row['here_boarding_point']
        
        # Set common REVIEW_REVIEWER
        REVIEW_REVIEWER.append("Recovery Script")

        # Check if ORIGIN or DESTIN are in a body of water using global_land_mask, if they are not None
        is_origin_water = not globe.is_land(*ORIGIN) if ORIGIN is not None else False
        is_destin_water = not globe.is_land(*DESTIN) if DESTIN is not None else False
        FLAG_WATER_ORIGIN.append(is_origin_water)
        FLAG_WATER_DESTIN.append(is_destin_water)
       

        # Three matching points
        if HOME == ORIGIN and HOME == DESTIN and ORIGIN == DESTIN:
            REVIEW_USAGE.append('Remove')
            FLAG_ALL_EQUAL.append(True)
            FLAG_POSS_HD.append(False)
            FLAG_POSS_TRAN.append(False)
        elif is_origin_water or is_destin_water:
            REVIEW_USAGE.append('REVIEW')
            FLAG_ALL_EQUAL.append(False)
            FLAG_POSS_HD.append(False)
            FLAG_POSS_TRAN.append(False)
        # Home possible origin or destination
        elif ORIGIN == DESTIN and (HOME != ORIGIN or HOME != DESTIN) and (HOME != None) and (DESTIN != None):
            REVIEW_USAGE.append('REVIEW')  # 'Use' changed to 'REVIEW'
            FLAG_ALL_EQUAL.append(False)
            FLAG_POSS_HD.append(True)
            FLAG_POSS_TRAN.append(False)
        
        # Possible transfer case
        else:
            # 'Use' changed to 'REVIEW'
            FLAG_ALL_EQUAL.append(False)
            FLAG_POSS_HD.append(False)
            
            if BOARDING is not None and haversine_distance(ORIGIN, BOARDING) > 2:
                FLAG_POSS_TRAN.append(True)
                REVIEW_USAGE.append('REVIEW')
            else:
                FLAG_POSS_TRAN.append(False)
                REVIEW_USAGE.append('Remove')
             

    # Add new columns to the DataFrame
    df['REVIEW_REVIEWER'] = REVIEW_REVIEWER
    df['REVIEW_USAGE'] = REVIEW_USAGE
    df['FLAG_ALL_EQUAL'] = FLAG_ALL_EQUAL
    df['FLAG_POSS_HD'] = FLAG_POSS_HD
    df['FLAG_POSS_TRAN'] = FLAG_POSS_TRAN
    df['FLAG_WATER_ORIGIN'] = FLAG_WATER_ORIGIN  # Adding new flag column for ORIGIN
    df['FLAG_WATER_DESTIN'] = FLAG_WATER_DESTIN  # Adding new flag column for DESTIN
    flag_columns = ['FLAG_ALL_EQUAL', 'FLAG_POSS_HD', 'FLAG_POSS_TRAN', 'FLAG_WATER_ORIGIN', 'FLAG_WATER_DESTIN']

    # Convert each flag column to integer type
    for col in flag_columns:
        df[col] = df[col].astype(int)
    # Select only the columns you want to keep in the final DataFrame
    return df[['REVIEW_REVIEWER', 'REVIEW_USAGE', 'Date_started', 'id', 'FLAG_ALL_EQUAL', 'FLAG_POSS_HD', 'FLAG_POSS_TRAN', 'FLAG_WATER_ORIGIN', 'FLAG_WATER_DESTIN']]


In [40]:
#recovery_df = (evaluate_removal_conditions(bp_df))

In [41]:
def make_survey_recovery_sheet(excel_path, df):
    # Generate a unique name for the new workbook using a timestamp
    #current_time = datetime.datetime.now().strftime("%Y%m%d%H%M%S")
    
    excel_path = f"{excel_path}.xlsx"
    
    # Create a Pandas Excel writer using XlsxWriter as the engine.
    with pd.ExcelWriter(excel_path, engine='xlsxwriter') as writer:
        # Write the DataFrame to the Excel workbook
        df.to_excel(writer, sheet_name='_(F0) SURVEY RECOVERY')
    # Close the Pandas Excel writer and output the Excel file.

#make_survey_recovery_sheet('hrtva_survey_recovery.xlsx', recovery_df)
    

In [42]:
# origin_latlong: 'ORIGIN_ADDRESS_LAT', 'ORIGIN_ADDRESS_LONG'
# destin_latlong: 'DESTIN_ADDRESS_LAT', 'ORIGIN_ADDRESS_LONG'

In [43]:
# Get boarding points, evaluate removal conditions, and create a survey recovery sheet for each municipality
for df, removals_df, name in zip([gcrta_df, hrtva_df, lacmta_df, bct_df, wake_df, victor_df, cota_df], [gcrta_removals, hrtva_removals, lacmta_removals, bct_removals, wake_removals, victor_removals, cota_removals], ['GCRTA', 'HRTVA', 'LACMTA', 'BCT','WAKE', 'VICTOR', 'COTA']):
    bp_df = get_boarding_points(removals_df)
    recovery_df = evaluate_removal_conditions(bp_df)
    make_survey_recovery_sheet(f'survey_recovery_sheets2/{name}_survey_recovery_{today}', recovery_df)
# for df, removals_df, name in zip([bct_df], [bct_removals], ['BCT']):
#     bp_df = get_boarding_points(removals_df, here_key)
#     recovery_df = evaluate_removal_conditions(bp_df)
#     make_survey_recovery_sheet(f'survey_recovery_sheets/{name}_survey_recovery.xlsx', recovery_df)

Processing Routes...


/tmp/ipykernel_263560/1374924134.py:70: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cord_df['here_boarding_point'] = here_boarding_points


13.389498393830518
13.684617449274292
8.400155513263147
5.145186504939579
1.54356834889599
0.55141659835871
0.0
0.0
6.096653802540503
6.196751359839865
3.311451163615425
5.336171500665792
0.6738515436647367
0.0
3.157629136035291
0.7478389728084031
0.5732748654593706


/tmp/ipykernel_263560/374032305.py:68: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['REVIEW_REVIEWER'] = REVIEW_REVIEWER
/tmp/ipykernel_263560/374032305.py:69: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['REVIEW_USAGE'] = REVIEW_USAGE
/tmp/ipykernel_263560/374032305.py:70: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stab

Processing Routes...
Processing Routes...


...

/tmp/ipykernel_263560/1374924134.py:70: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cord_df['here_boarding_point'] = here_boarding_points
/tmp/ipykernel_263560/374032305.py:68: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['REVIEW_REVIEWER'] = REVIEW_REVIEWER
/tmp/ipykernel_263560/374032305.py:69: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata

Processing Routes...
0.0


/tmp/ipykernel_263560/1374924134.py:70: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cord_df['here_boarding_point'] = here_boarding_points
/tmp/ipykernel_263560/374032305.py:68: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['REVIEW_REVIEWER'] = REVIEW_REVIEWER
/tmp/ipykernel_263560/374032305.py:69: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata

5.500260434879957
16.35936401914248
4.244327424323096
8.76755449222564
4.209220420610854
3.0055587704251305
5.733264188136184
Processing Routes...
Processing Routes...


/tmp/ipykernel_263560/1374924134.py:70: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cord_df['here_boarding_point'] = here_boarding_points
/tmp/ipykernel_263560/374032305.py:68: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['REVIEW_REVIEWER'] = REVIEW_REVIEWER
/tmp/ipykernel_263560/374032305.py:69: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata

In [44]:
#make_survey_recovery_sheet('LACMTA_survey_recovery.xlsx', recovery_df)

# Fixing past recoveries

In [45]:
#df = pd.read_excel('survey_recovery_sheets2/GCRTA_survey_recovery.xlsx')
#df = pd.read_excel('survey_recovery_sheets2/HRTVA_survey_recovery.xlsx')
# df = pd.read_excel('survey_recovery_sheets2/LACMTA_survey_recovery.xlsx')
municipality = 'COTA'
df = pd.read_excel(f'survey_recovery_sheets2/{municipality}_survey_recovery_{today}.xlsx')

In [46]:


# if REVIEW_USAGE == 'Remove' then drop that row
if 'REVIEW_USAGE' in df.columns:
    df = df[df['REVIEW_USAGE'] != 'Remove']
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])

# Rename REVIEW_REVIEWER to 1st Reviewer
df = df.rename(columns={'REVIEW_REVIEWER': '1st Reviewer'})
# Rename REVIEW_USAGE to 1st Reviewer Usage
df = df.rename(columns={'REVIEW_USAGE': '1st Reviewer Usage'})

#For df, add ROUTE_SURVEYEDCode from bct_ke
df = df.merge(cota_ke[['id', 'ROUTE_SURVEYEDCode']], on='id', how='left')

#For df, add ELVIS_STATUS from bct_ke
df = df.merge(cota_ke[['id', 'ELVIS_STATUS']], on='id', how='left')



# Add a column at the beginning of the DataFrame for the 2nd Reviewer
df.insert(0, '2nd Reviewer', None)
# Add a column at the beginning of the DataFrame for the 2nd Reviewer Usage 
df.insert(1, '2nd Reviewer Usage', None)
df.insert(2, 'Final_Usage', None)

# Move ROUTE_SURVEYEDCode to the beginning of the dataframe
df = df[['ROUTE_SURVEYEDCode'] + [col for col in df.columns if col != 'ROUTE_SURVEYEDCode']]

# Make "ELVIS_STATUS" the sixth column of the dataframe


# Remove duplicates from the dataframe based on elvis_id, while removing the first instance
df = df.drop_duplicates(subset=['id'], keep='last')





In [47]:
upper_municipality = municipality.upper()

In [48]:
df.to_csv(f'revised_recovery/{upper_municipality}_KINGElvis_recovery_revised_{today}.csv', index=False)